# BAREC Dataset Arabic Processing Configuration Demo

This notebook demonstrates how different Arabic text processing configurations affect the BAREC dataset examples. We'll test various normalization, diacritization, and morphological processing options and visualize the results.

## Setup and Configuration

In [1]:
import sys
import os
import logging
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, List, Any
import torch
from PIL import Image
import numpy as np
from collections import defaultdict

# Add the project root to Python path
sys.path.append('/home/bens/pixel')

from src.pixel.data.datasets.barec_dataset import BARECDataset
from src.pixel.data.rendering import PangoCairoTextRenderer, PyGameTextRenderer
from src.pixel import Modality, get_transforms
from src.pixel.data.processing.arabic_sentence_processor import (
    ProcessingConfig, 
    ArabicSentenceProcessor,
    OrthographicFormat,
    DiacriticFormat,
    MorphologicalScheme,
    EncodingScheme,
    create_default_config,
    create_normalized_config,
    create_diacritized_config,
    create_morphological_config,
    create_buckwalter_config
)

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ All imports successful!")

/opt/anaconda3/envs/pixel-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful!


## Define Processing Configurations

Let's create various processing configurations to test:

In [2]:
from src.pixel.data.processing.experiment_configs import ALL_CONFIGS
# Define various processing configurations for testing
processing_configs = ALL_CONFIGS

print(f"Created {len(processing_configs)} processing configurations:")
for name in processing_configs.keys():
    print(f"  - {name}")

Created 20 processing configurations:
  - no-unicode-normalize
  - arabic-isolated
  - arabic-default
  - arabic-dediac
  - arabic-norm
  - arabic-norm-dediac
  - arabic-nonorm-diac
  - buckwalter-default
  - buckwalter-norm-dediac
  - buckwalter-nonorm-diac
  - hsb-default
  - hsb-nonorm-diac
  - hsb-norm-dediac
  - morph-word
  - morph-lex
  - morph-d3tok-default
  - morph-d3tok-tatweel
  - morph-d3tok-tatweel2
  - morph-d3tok-tatweel3
  - morph-d3tok-space


## Register Configurations and Test Processing

First, let's test the Arabic sentence processor with some sample sentences:

In [3]:
if False:
    # Test sample sentences
    sample_sentences = [
        "هَـــلْ ذَهَبْتَ إِلَى المَكْتَبَةِ؟",  # With diacritics and tatweel
        "الولايات المتحدة الأمريكية دولة كبيرة",  # Complex phrase
        "كَتَبَ الطالِبُ الدَّرْسَ بِعِنايَةٍ",  # Diacritized sentence
        "في هذا اليوم الجميل نذهب إلى المدرسة"  # Simple sentence
    ]

    print("\n🔍 Testing sentence processing with different configurations:")
    print("=" * 80)

    # Test each configuration with the first sample sentence
    test_sentence = sample_sentences[0]
    print(f"Original: {test_sentence}")
    print("-" * 50)

    processing_results = {}
    for config_name, config in processing_configs.items():
        try:
            processor = ArabicSentenceProcessor(config)
            result = processor.process(test_sentence)
            processing_results[config_name] = result
            print(f"{config_name:15}: {result}")
        except Exception as e:
            print(f"{config_name:15}: ERROR - {e}")
            processing_results[config_name] = f"ERROR: {e}"

## Load BAREC Dataset with Different Configurations

Now let's load the BAREC dataset with different processing configurations:

In [4]:
# Configuration for dataset loading
DATASET_CONFIG = {
    "dataset_name": "CAMeL-Lab/BAREC-Shared-Task-2025-sent",
    "split": "test",  # Use train split for examples
    "max_seq_length": 256,
    "renderer_path": "Team-PIXEL/pixel-base",
    "num_samples": 10  # Load only first 10 samples for demo
}

print(f"📊 Loading BAREC dataset: {DATASET_CONFIG['dataset_name']}")
print(f"Split: {DATASET_CONFIG['split']}, Max length: {DATASET_CONFIG['max_seq_length']}")

📊 Loading BAREC dataset: CAMeL-Lab/BAREC-Shared-Task-2025-sent
Split: test, Max length: 256


In [5]:
# Load renderer for image generation
print("🔧 Loading PIXEL renderer...")
renderer = PangoCairoTextRenderer.from_pretrained(DATASET_CONFIG["renderer_path"])
renderer.max_seq_length = DATASET_CONFIG["max_seq_length"]

# Set up transforms
transforms = get_transforms(
    do_resize=True,
    size=(renderer.pixels_per_patch, renderer.pixels_per_patch * renderer.max_seq_length),
)

print(f"   ✅ Renderer loaded: {renderer.pixels_per_patch}px per patch")
print(f"   ✅ Transform size: {transforms.transforms[0].size if hasattr(transforms.transforms[0], 'size') else 'N/A'}")

🔧 Loading PIXEL renderer...


INFO:src.pixel.data.rendering.rendering_utils:loading text renderer configuration file https://huggingface.co/Team-PIXEL/pixel-base/resolve/main/text_renderer_config.json from cache at /shared_models/.cache/huggingface/transformers/892d6a02d7c441000de399de59ed70d943a81f7b0f536523b4af1111677a8508.e332b34c9c05756dd4aa51d8fa33461dbd79604752296d185f03f8004db30700
INFO:src.pixel.data.rendering.rendering_utils:loading font file https://huggingface.co/Team-PIXEL/pixel-base/resolve/main/GoNotoCurrent.ttf from cache at /shared_models/.cache/huggingface/transformers/49e6dc219d1a1a1c9236acaf05a48b542002016a6dc877ee72baab085a84257b.3f28e7f4b38e1efe1b6da4a3732404c19d4c6a614ff32dce90a251e293d4ce58
INFO:src.pixel.data.rendering.pangocairo_renderer:Loading font from /shared_models/.cache/huggingface/transformers/49e6dc219d1a1a1c9236acaf05a48b542002016a6dc877ee72baab085a84257b.3f28e7f4b38e1efe1b6da4a3732404c19d4c6a614ff32dce90a251e293d4ce58
INFO:src.pixel.data.rendering.rendering_utils:Text renderer Pa

   ✅ Renderer loaded: 16px per patch
   ✅ Transform size: N/A


In [7]:
# Load datasets with different processing configurations
datasets = {}
sample_data = {}

# Select a subset of configurations for dataset loading (to avoid loading too many)
# selected_configs = ["arabic-default", "arabic-norm", "arabic-nonorm-diac", "morph-d3tok-tatweel","morph-d3tok-default", "buckwalter-default", "hsb-default", "hsb-nonorm-diac"]
selected_configs = ["arabic-default", "arabic-nonorm-diac", "buckwalter-default", "buckwalter-nonorm-diac", "hsb-default", "hsb-nonorm-diac"]

print("📦 Loading datasets with different processing configurations...")

for config_name in selected_configs:
    try:
        print(f"\n   Loading: {config_name}")
        
        # Create dataset
        dataset = BARECDataset(
            dataset_name=DATASET_CONFIG["dataset_name"],
            processor=renderer,
            modality=Modality.IMAGE,
            max_seq_length=DATASET_CONFIG["max_seq_length"],
            split=DATASET_CONFIG["split"],
            transforms=transforms,
            processing_config_name=config_name
        )
        
        datasets[config_name] = dataset
        
        # Extract sample data for analysis
        sample_data[config_name] = {
            'sentences': [dataset.examples[i].sentence for i in range(min(DATASET_CONFIG["num_samples"], len(dataset)))],
            'labels': [dataset.examples[i].label for i in range(min(DATASET_CONFIG["num_samples"], len(dataset)))],
            'ids': [dataset.examples[i].id for i in range(min(DATASET_CONFIG["num_samples"], len(dataset)))]
        }
        
        print(f"   ✅ Loaded {len(dataset)} examples")
        
    except Exception as e:
        print(f"   ❌ Error loading {config_name}: {e}")
        datasets[config_name] = None
        sample_data[config_name] = None

print(f"\n✅ Successfully loaded {len([d for d in datasets.values() if d is not None])} datasets")

INFO:src.pixel.data.datasets.barec_dataset:Creating features from HuggingFace dataset (no cache)


📦 Loading datasets with different processing configurations...

   Loading: arabic-default


INFO:src.pixel.data.datasets.barec_dataset:Initialized Arabic sentence processor with config: arabic-default
100%|██████████| 7286/7286 [00:01<00:00, 7263.36it/s]
/home/bens/pixel/src/pixel/data/datasets/barec_dataset.py:71: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
INFO:src.pixel.data.datasets.barec_dataset:*** Example ***
INFO:src.pixel.data.datasets.barec_dataset:sentence: ماجد
INFO:src.pixel.data.datasets.barec_dataset:attention_mask: tensor([1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.,

   ✅ Loaded 7286 examples

   Loading: arabic-nonorm-diac


INFO:src.pixel.data.processing.arabic_sentence_processor:Loading CAMeL Tools disambiguator...
INFO:src.pixel.data.datasets.barec_dataset:Initialized Arabic sentence processor with config: arabic-nonorm-diac
100%|██████████| 7286/7286 [01:40<00:00, 72.47it/s]
INFO:src.pixel.data.datasets.barec_dataset:*** Example ***
INFO:src.pixel.data.datasets.barec_dataset:sentence: ماجِد
INFO:src.pixel.data.datasets.barec_dataset:attention_mask: tensor([1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
    

...

   ✅ Loaded 7286 examples

   Loading: hsb-default


INFO:src.pixel.data.datasets.barec_dataset:Initialized Arabic sentence processor with config: hsb-default
100%|██████████| 7286/7286 [00:01<00:00, 4308.75it/s]
INFO:src.pixel.data.datasets.barec_dataset:*** Example ***
INFO:src.pixel.data.datasets.barec_dataset:sentence: mAjd
INFO:src.pixel.data.datasets.barec_dataset:attention_mask: tensor([1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 

   ✅ Loaded 7286 examples

   Loading: hsb-nonorm-diac


INFO:src.pixel.data.processing.arabic_sentence_processor:Loading CAMeL Tools disambiguator...
INFO:src.pixel.data.datasets.barec_dataset:Initialized Arabic sentence processor with config: hsb-nonorm-diac
100%|██████████| 7286/7286 [01:41<00:00, 71.94it/s]
INFO:src.pixel.data.datasets.barec_dataset:*** Example ***
INFO:src.pixel.data.datasets.barec_dataset:sentence: mAjid
INFO:src.pixel.data.datasets.barec_dataset:attention_mask: tensor([1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       

   ✅ Loaded 7286 examples

✅ Successfully loaded 6 datasets


## Compare Processing Results

Let's create a comparison table showing how the same sentences are processed differently:

In [8]:
# Create comparison DataFrame
comparison_data = []

for i in range(DATASET_CONFIG["num_samples"]):
    row = {'Example': i + 1}
    
    for config_name in selected_configs:
        if sample_data[config_name] is not None:
            sentence = sample_data[config_name]['sentences'][i]
            label = sample_data[config_name]['labels'][i]
            row[f'{config_name}'] = sentence
            row[f'{config_name}_label'] = label
    
    comparison_data.append(row)

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_data)

print("📋 Sentence Processing Comparison (First 5 Examples)")
print("=" * 100)

for i, row in comparison_df.iterrows():
    print(f"\n🔍 Example {row['Example']}:")
    print("-" * 50)
    
    for config_name in selected_configs:
        if config_name in row and pd.notna(row[config_name]):
            sentence = row[config_name]
            label = row.get(f'{config_name}_label', 'N/A')
            print(f"{config_name:15}: {sentence} (Label: {label})")

📋 Sentence Processing Comparison (First 5 Examples)

🔍 Example 1:
--------------------------------------------------
arabic-default : ماجد (Label: 0)
arabic-nonorm-diac: ماجِد (Label: 0)
buckwalter-default: mAjd (Label: 0)
buckwalter-nonorm-diac: mAjid (Label: 0)
hsb-default    : mAjd (Label: 0)
hsb-nonorm-diac: mAjid (Label: 0)

🔍 Example 2:
--------------------------------------------------
arabic-default : الأربعاء 15 يونيو 2011م (Label: 12)
arabic-nonorm-diac: الأَرْبِعاءَ 15 يُونِيُو 2011م (Label: 12)
buckwalter-default: Al>rbEA' 15 ywnyw 2011m (Label: 12)
buckwalter-nonorm-diac: Al>arobiEA'a 15 yuwniyuw 2011m (Label: 12)
hsb-default    : AlÂrbςA' 15 ywnyw 2011m (Label: 12)
hsb-nonorm-diac: AlÂar.biςA'a 15 yuwniyuw 2011m (Label: 12)

🔍 Example 3:
--------------------------------------------------
arabic-default : – 13 رجب 1432ه - (Label: 11)
arabic-nonorm-diac: –13 رَجَب 1432ه- (Label: 11)
buckwalter-default: – 13 rjb 1432h - (Label: 11)
buckwalter-nonorm-diac: –13 rajab 1432h- (L

## Visualize Rendered Images

Let's visualize how the different processing configurations affect the rendered images:

In [9]:
datasets

{'arabic-default': <src.pixel.data.datasets.barec_dataset.BARECDataset at 0x7f6306da7a90>,
 'arabic-nonorm-diac': <src.pixel.data.datasets.barec_dataset.BARECDataset at 0x7f6306da7b20>,
 'buckwalter-default': <src.pixel.data.datasets.barec_dataset.BARECDataset at 0x7f62de147a00>,
 'buckwalter-nonorm-diac': <src.pixel.data.datasets.barec_dataset.BARECDataset at 0x7f6160eeda00>,
 'hsb-default': <src.pixel.data.datasets.barec_dataset.BARECDataset at 0x7f6160d8a880>,
 'hsb-nonorm-diac': <src.pixel.data.datasets.barec_dataset.BARECDataset at 0x7f6160dda3a0>}

In [10]:
def tensor_to_numpy(pixel_values):
    if isinstance(pixel_values, torch.Tensor):
        if pixel_values.dim() == 3:  # [C, H, W]
            if pixel_values.shape[0] == 3:  # RGB
                return pixel_values.permute(1, 2, 0).cpu().numpy()
            else:  # Grayscale
                return pixel_values[0].cpu().numpy()
        else:
            return pixel_values.cpu().numpy()
    return np.array(pixel_values)

def crop_to_text_width(img_array, num_patches, patch_width=16):
    text_width = num_patches * patch_width
    if img_array.shape[1] > text_width:
        if len(img_array.shape) == 2:
            return img_array[:, :text_width]
        else:
            return img_array[:, :text_width, :]
    return img_array

def add_patch_boundaries(img_array, num_patches, patch_width=16):
    # Create a copy to avoid modifying the original
    img_with_boundaries = img_array.copy()
    text_width = num_patches * patch_width
    
    # Add dashed vertical lines at patch boundaries
    for patch_start in range(patch_width, text_width, patch_width):
        if patch_start < img_array.shape[1]:
            # Create dashed line pattern (every 3rd pixel) with alpha blending
            for y in range(0, img_array.shape[0], 3):
                if y < img_array.shape[0]:
                    if len(img_array.shape) == 2:  # Grayscale
                        # Alpha blend with existing pixel
                        original = img_with_boundaries[y, patch_start]
                        img_with_boundaries[y, patch_start] = 0.5 * 0.5 + original * 0.5
                    else:  # RGB
                        # Alpha blend with existing pixel
                        original = img_with_boundaries[y, patch_start, :]
                        line_color = np.array([1, 0.4, 0.4])
                        img_with_boundaries[y, patch_start, :] = line_color * 0.5 + original * 0.5

    return img_with_boundaries

def save_rendered_image(dataset, config_name, example_idx, save_dir="rendered_images"):
    os.makedirs(save_dir, exist_ok=True)
    
    try:
        example = dataset[example_idx]
        pixel_values = example['pixel_values']
        num_patches = example['attention_mask'].sum()
        
        img_array = tensor_to_numpy(pixel_values)
        img_array = crop_to_text_width(img_array, num_patches)
        img_array = add_patch_boundaries(img_array, num_patches)
        
        # Convert to PIL Image for saving
        if len(img_array.shape) == 2:  # Grayscale
            img = Image.fromarray((img_array * 255).astype(np.uint8), mode='L')
        else:  # RGB
            img = Image.fromarray((img_array * 255).astype(np.uint8))
        
        filename = f"{example_idx:03d}_{config_name}.png"
        filepath = os.path.join(save_dir, filename)
        img.save(filepath)
        
        return filepath
        
    except Exception as e:
        print(f"Error saving {config_name} example {example_idx}: {e}")
        return None

def plot_rendered_images(datasets, example_idx=0, save_dir="rendered_images"):
    saved_files = []
    
    for config_name, dataset in datasets.items():
        if dataset is not None:
            filepath = save_rendered_image(dataset, config_name, example_idx, save_dir)
            if filepath:
                saved_files.append(filepath)
                print(f"Saved: {filepath}")
    
    print(f"\nSaved {len(saved_files)} images to {save_dir}/")
    return saved_files

# Usage
plot_rendered_images(datasets, example_idx=230)

Saved: rendered_images/230_arabic-default.png
Saved: rendered_images/230_arabic-nonorm-diac.png
Saved: rendered_images/230_buckwalter-default.png
Saved: rendered_images/230_buckwalter-nonorm-diac.png
Saved: rendered_images/230_hsb-default.png
Saved: rendered_images/230_hsb-nonorm-diac.png

Saved 6 images to rendered_images/


['rendered_images/230_arabic-default.png',
 'rendered_images/230_arabic-nonorm-diac.png',
 'rendered_images/230_buckwalter-default.png',
 'rendered_images/230_buckwalter-nonorm-diac.png',
 'rendered_images/230_hsb-default.png',
 'rendered_images/230_hsb-nonorm-diac.png']

In [11]:
# Plot images for another example
print("🖼️ Rendered Images Comparison (Example 2)")
for idx in [2462, 434, 508, 648, 893, 1876, 2413, 2447, 5718, 5719]:
    plot_rendered_images(datasets, example_idx=idx)

🖼️ Rendered Images Comparison (Example 2)
Saved: rendered_images/2462_arabic-default.png
Saved: rendered_images/2462_arabic-nonorm-diac.png
Saved: rendered_images/2462_buckwalter-default.png
Saved: rendered_images/2462_buckwalter-nonorm-diac.png
Saved: rendered_images/2462_hsb-default.png
Saved: rendered_images/2462_hsb-nonorm-diac.png

Saved 6 images to rendered_images/
Saved: rendered_images/434_arabic-default.png
Saved: rendered_images/434_arabic-nonorm-diac.png
Saved: rendered_images/434_buckwalter-default.png
Saved: rendered_images/434_buckwalter-nonorm-diac.png
Saved: rendered_images/434_hsb-default.png
Saved: rendered_images/434_hsb-nonorm-diac.png

Saved 6 images to rendered_images/
Saved: rendered_images/508_arabic-default.png
Saved: rendered_images/508_arabic-nonorm-diac.png
Saved: rendered_images/508_buckwalter-default.png
Saved: rendered_images/508_buckwalter-nonorm-diac.png
Saved: rendered_images/508_hsb-default.png
Saved: rendered_images/508_hsb-nonorm-diac.png

Saved 6 i

In [12]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import math
import os


def plot_dataset_examples(dataset, batch_size=16, crop_width=400, save_dir=None, prefix="batch", n_batches=None):
    """
    Render all examples in a dataset in batches with tight vertical stacking.
    
    Parameters:
    - dataset: Dataset with 'pixel_values' field
    - batch_size: Number of examples per batch (default 16)
    - crop_width: Maximum width in pixels (default 400)
    - save_dir: Directory to save PNG files (if None, displays instead)
    - prefix: Filename prefix for saved images
    - n_batches: Limit number of batches (if None, processes all)
    """
    n_examples = len(dataset)
    if n_batches is None:
        n_batches = math.ceil(n_examples / batch_size)
    else:
        n_batches = min(n_batches, math.ceil(n_examples / batch_size))

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    idx = 0
    batch_count = 0

    while batch_count < n_batches and idx < n_examples:  # Added bounds check

        # Process all images first to determine dimensions
        processed_images = []
        max_height = 0
        
        included_count = 0
        start_idx = idx  # Track starting index for this batch

        # Try to fill a batch, but don't loop forever
        attempts = 0
        max_attempts = min(batch_size * 10, n_examples - idx)  # Reasonable limit
        
        while included_count < batch_size and idx < n_examples and attempts < max_attempts:
            try:
                example = dataset[idx]
                attempts += 1

                # Filter based on attention mask
                if example["attention_mask"].sum() > 26 or example["attention_mask"].sum() < 8:
                    idx += 1
                    continue

                current_idx = idx  # Store current index before incrementing
                included_count += 1
                idx += 1

                pixel_values = example['pixel_values']

                # Convert to numpy array
                if isinstance(pixel_values, torch.Tensor):
                    if pixel_values.dim() == 3:  # [C, H, W]
                        if pixel_values.shape[0] == 3:  # RGB
                            img_array = pixel_values.permute(1, 2, 0).cpu().numpy()
                        elif pixel_values.shape[0] == 1:  # Grayscale with channel dim
                            img_array = pixel_values[0].cpu().numpy()
                        else:  # Handle other channel configurations
                            img_array = pixel_values[0].cpu().numpy()
                    else:  # 2D tensor
                        img_array = pixel_values.cpu().numpy()
                else:
                    img_array = np.array(pixel_values)

                # Ensure proper format
                if img_array.ndim == 3 and img_array.shape[2] == 1:
                    img_array = img_array.squeeze(axis=2)

                # Crop width if necessary
                if crop_width is not None and img_array.shape[1] > crop_width:
                    if img_array.ndim == 2:
                        img_array = img_array[:, :crop_width]
                    else:
                        img_array = img_array[:, :crop_width, :]

                processed_images.append((current_idx, img_array))
                max_height = max(max_height, img_array.shape[0])
                
            except Exception as e:
                print(f"Error processing example {idx}: {e}")
                processed_images.append((idx, None))
                idx += 1  # Move to next example even on error

        # If we couldn't find enough valid examples, break
        if not processed_images:
            print(f"No valid examples found for batch {batch_count + 1}")
            break
            
        # If we have fewer than expected, that's okay - just process what we have
        if included_count < batch_size:
            print(f"Batch {batch_count + 1} only has {included_count} examples (filtered dataset)")

        # Create figure with tight spacing
        total_height = sum(img[1].shape[0] if img[1] is not None else max_height 
                          for img in processed_images)
        
        if total_height == 0:  # Safety check
            print(f"Skipping batch {batch_count + 1} - no valid images")
            batch_count += 1
            continue
        
        # Calculate figure dimensions - make it narrow and tall
        fig_width = 8  # Fixed width for consistent appearance
        fig_height = max(6, total_height / 50)  # Scale height based on content
        
        fig = plt.figure(figsize=(fig_width, fig_height))
        fig.patch.set_facecolor('white')
        
        # Remove all margins and padding
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0, hspace=0, wspace=0)
        
        current_y = 1.0  # Start from top
        
        for img_data in processed_images:
            ex_idx, img_array = img_data
            
            if img_array is not None:
                img_height = img_array.shape[0]
                row_height = img_height / total_height
                
                # Create subplot with exact positioning
                ax = fig.add_axes([0.08, current_y - row_height, 0.88, row_height])
                
                # Display image
                if img_array.ndim == 2:  # Grayscale
                    ax.imshow(img_array, cmap='gray', aspect='auto', interpolation='nearest')
                else:  # Color
                    ax.imshow(img_array, aspect='auto', interpolation='nearest')
                
                # Remove all axis elements
                ax.set_xticks([])
                ax.set_yticks([])
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.spines['bottom'].set_visible(False)
                ax.spines['left'].set_visible(False)
                
                # Add index label on the left margin
                fig.text(0.02, current_y - row_height/2, str(ex_idx), 
                        fontsize=8, ha='center', va='center', 
                        bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.8))
                
            else:
                # Handle error cases
                row_height = max_height / total_height if total_height > 0 else 0.1
                ax = fig.add_axes([0.08, current_y - row_height, 0.88, row_height])
                ax.text(0.5, 0.5, f"Error: {ex_idx}", ha='center', va='center', 
                       fontsize=8, transform=ax.transAxes)
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1)
                ax.axis('off')
                
                fig.text(0.02, current_y - row_height/2, str(ex_idx), 
                        fontsize=8, ha='center', va='center',
                        bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.8))
            
            current_y -= row_height

        # Save or display
        if save_dir:
            out_path = os.path.join(save_dir, f"{prefix}_{batch_count:03d}.png")
            plt.savefig(out_path, dpi=150, bbox_inches='tight', pad_inches=0, 
                       facecolor='white', edgecolor='none')
            plt.close(fig)
            print(f"Saved batch {batch_count + 1}/{n_batches}: {out_path}")
        else:
            plt.show()

        batch_count += 1


# Example usage
plot_dataset_examples(
    dataset=datasets["arabic-default"],
    batch_size=16,
    crop_width=400,
    save_dir="rendered_examples_test",
    prefix="arabic-default", 
    n_batches=300
)

Saved batch 1/300: rendered_examples_test/arabic-default_000.png
Saved batch 2/300: rendered_examples_test/arabic-default_001.png
Saved batch 3/300: rendered_examples_test/arabic-default_002.png
Saved batch 4/300: rendered_examples_test/arabic-default_003.png
Saved batch 5/300: rendered_examples_test/arabic-default_004.png
Saved batch 6/300: rendered_examples_test/arabic-default_005.png
Saved batch 7/300: rendered_examples_test/arabic-default_006.png
Saved batch 8/300: rendered_examples_test/arabic-default_007.png
Saved batch 9/300: rendered_examples_test/arabic-default_008.png
Saved batch 10/300: rendered_examples_test/arabic-default_009.png
Saved batch 11/300: rendered_examples_test/arabic-default_010.png
Saved batch 12/300: rendered_examples_test/arabic-default_011.png
Saved batch 13/300: rendered_examples_test/arabic-default_012.png
Saved batch 14/300: rendered_examples_test/arabic-default_013.png
Saved batch 15/300: rendered_examples_test/arabic-default_014.png
Saved batch 16/300: